In [2]:
import os
import math
import time
import operator
import datetime
import h5py
from tqdm import tqdm
import wandb

import numpy as np
from scipy.ndimage import rotate
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.image as image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.modules.utils import _pair, _quadruple
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset
import torchvision

import sys
sys.path.append('/scratch/yichen/notebooks/helper_functions/')
from visualization_functions import show_images


class hdf5_dataset(Dataset):
    
    def __init__(self, file_path, folder='train', transform=None, 
                 classes=[], mode='single'):
        self.file_path = file_path
        self.folder = folder
        self.transform = transform
        self.classes = classes
        self.mode = mode
        self.hf = None

    def __len__(self):
        with h5py.File(self.file_path, 'r') as f:
            self.len = len(f[self.folder][self.folder+'_labels'])
        return self.len
    
    def __getitem__(self, idx):
        if self.hf is None:
            self.hf = h5py.File(self.file_path, 'r')

        label = np.array(self.hf[self.folder][self.folder+'_labels'][idx])
        metadata = np.array(self.hf[self.folder][self.folder+'_metadata'][idx])
        
        if self.mode == 'single':
            unit_cell = np.array(self.hf[self.folder][self.folder+'_unit_cell'][idx])

            image = np.array(self.hf[self.folder][self.folder+'_data'][idx])
            if self.transform:
                image = self.transform(image)
            return image, label, unit_cell, metadata
    
        if self.mode == 'pair':
            unit_cell1 = np.array(self.hf[self.folder][self.folder+'_unit_cell1'][idx])
            unit_cell2 = np.array(self.hf[self.folder][self.folder+'_unit_cell2'][idx])
            
            image1 = np.array(self.hf[self.folder][self.folder+'_data1'][idx])
            image2 = np.array(self.hf[self.folder][self.folder+'_data2'][idx])
            if self.transform:
                image1 = self.transform(image1)
                image2 = self.transform(image2)
            return [image1, image2], label, [unit_cell1, unit_cell2], metadata
        
class process_function():
    def __init__(self, input_index, label_index, device=torch.device('cpu')):
        self.input_index = input_index
        self.label_index = label_index
        self.device = device

    def process(self, batch):
        inputs = batch[self.input_index]
        if isinstance(inputs, list):
            for i, d in enumerate(inputs):
                inputs[i] = d.float().to(self.device)
        else:
            inputs = inputs.float().to(self.device)
                
        labels = batch[self.label_index].long().to(self.device)
        return inputs, labels

In [3]:
symmetry_label_dict = {'p1':0, 'p2':1, 'pm':2, 'p4':3, 'p3':4}

imagenet_path = '/scratch/yichen/imagenet_atom_p1p2pmp4p3_pair_unrot.h5'
atom_path = '/scratch/yichen/imagenet_atom_p1p2pmp4p3_with_uc_unrot.h5'

bs = 312
shuffle = True

# imagenet
train_ds = hdf5_dataset(imagenet_path, folder='train', 
                        transform=transforms.ToTensor(), 
                        classes=symmetry_label_dict.keys(), mode='pair')
train_dl = DataLoader(train_ds, batch_size=bs, shuffle=shuffle, num_workers=4)

valid_ds = hdf5_dataset(imagenet_path, folder='valid', 
                        transform=transforms.ToTensor(), 
                        classes=symmetry_label_dict.keys(), mode='pair')
valid_dl = DataLoader(valid_ds, batch_size=bs, shuffle=shuffle, num_workers=4)

test_ds = hdf5_dataset(atom_path, folder='test', 
                        transform=transforms.ToTensor(), 
                        classes=symmetry_label_dict.keys())
test_dl = DataLoader(test_ds, batch_size=bs, shuffle=shuffle, num_workers=4)

In [4]:
# proc = process_function(0,1)    

# batch = next(iter(train_dl))
# inputs, labels = proc.process(batch)
# show_images(torch.permute(inputs[0], (0,2,3,1)).numpy())
# show_images(torch.permute(inputs[1], (0,2,3,1)).numpy())

# batch = next(iter(valid_dl))
# inputs, labels = proc.process(batch)
# show_images(torch.permute(inputs[0], (0,2,3,1)).numpy())
# show_images(torch.permute(inputs[1], (0,2,3,1)).numpy())

# batch = next(iter(test_dl))
# inputs, labels = proc.process(batch)
# show_images(torch.permute(inputs, (0,2,3,1)).numpy())

In [5]:
def train_epochs(model, loss_func_list, optimizer, device, train_dl, valid_dl, test_dl,
                 start, epochs, process_func, scheduler=None, save_per_epochs=10,
                 model_name=None, model_dir=None, tracking=False):

    # large value for initial best loss
    best_loss = 1e7
    best_acc = 0

    # make directory for the model
    if model_dir and not os.path.isdir(model_dir): os.mkdir(model_dir)

    
    if tracking:   
        wandb.watch(model, log_freq=100)
        
    # make first contrastive loss really small
    avg_train_loss_contrastive = 1e5
    avg_train_loss_xentropy = 1

    prev_coef_list = [1]
    for i, epoch_idx in enumerate(range(start, epochs+start)):
        
        print("Epoch: {}/{}".format(epoch_idx+1, epochs+start))
        
        contrastive_loss_func, prev_coef = loss_func_list[0].LossFunction(epoch_idx, prev_coef_list, avg_train_loss_contrastive, avg_train_loss_xentropy)
        loss_func_epoch = [contrastive_loss_func, loss_func_list[1]]
        prev_coef_list.append(prev_coef)
            
            
        out_train = train(model, loss_func_epoch, optimizer, device, train_dl, process_func,
                                              scheduler=None, tracking=tracking)
        avg_train_loss, avg_train_loss_contrastive, avg_train_loss_xentropy, avg_train_acc = out_train
        
        out_valid = valid(model, loss_func_epoch, device, valid_dl, process_func, test=False,
                                              tracking=tracking)
        avg_valid_loss, avg_valid_loss_contrastive, avg_valid_loss_xentropy, avg_valid_acc = out_valid
        
        if test_dl:
            avg_test_loss, avg_test_acc = valid(model, loss_func_epoch, device, test_dl, process_func, test=True,
                                                tracking=tracking)
            
        if tracking:   
            # record the epoch loss and accuracy:
            if test_dl:
                wandb.log({'epoch': epoch_idx, 
                           'prev_coef': prev_coef,
                           "train_loss": avg_train_loss, 
                           "train_loss_contrastive": avg_train_loss_contrastive, 
                           "train_loss_xentropy": avg_train_loss_xentropy, 
                           "train_acc": avg_train_acc, 
                           
                           "valid_loss": avg_valid_loss, 
                           "valid_loss_contrastive": avg_valid_loss_contrastive, 
                           "valid_loss_xentropy": avg_valid_loss_xentropy, 
                           "valid_acc": avg_valid_acc,
                           
                           "test_loss": avg_test_loss,
                           "test_acc": avg_test_acc})
            else:
                wandb.log({'epoch':epoch_idx, 
                           'prev_coef': prev_coef,
                           "train_loss": avg_train_loss, 
                           "train_loss_contrastive": avg_train_loss_contrastive, 
                           "train_loss_xentropy": avg_train_loss_xentropy, 
                           "train_acc": avg_train_acc, 
                           
                           "valid_loss": avg_valid_loss, 
                           "valid_loss_contrastive": avg_valid_loss_contrastive, 
                           "valid_loss_xentropy": avg_valid_loss_xentropy, 
                           "valid_acc": avg_valid_acc}) 
                                
        if model_name != None:            
            if (epoch_idx + 1) % save_per_epochs == 0:
                torch.save(model, os.path.join(model_dir, model_name+'-epoch-{}.pt'.format(epoch_idx+1)))

            if avg_valid_loss < best_loss:
                torch.save(model, os.path.join(model_dir, model_name+'-best_loss.pt'))

            if avg_valid_acc > best_acc:
                torch.save(model, os.path.join(model_dir, model_name+'-best_acc.pt'))
              
    return 

def train(model, loss_func_list, optimizer, device, train_dl, process_func, 
          scheduler=None, tracking=False):
    
    train_data_size = len(train_dl.dataset)
    model = model.to(device)

    # Set to training mode
    model.train()

    # Loss and Accuracy within the epoch
    train_loss = 0.0
    train_loss_contrastive = 0.0
    train_loss_xentropy = 0.0
    train_acc = 0.0

#     for i, batch in enumerate(train_dl):
    for i, batch in enumerate(tqdm(train_dl)):

        # Clean existing gradients
        optimizer.zero_grad()
        
        [inputs, inputs_p], labels = process_func.process(batch)

        # loss - contrasitve
        outputs_fpn, outputs = model(inputs)
        outputs_p_fpn, outputs_p = model(inputs_p)
        loss_contrastive = loss_func_list[0](outputs_fpn, outputs_p_fpn) 

        # loss 2
        loss_xentropy = loss_func_list[1](outputs, labels)

        loss = loss_contrastive + loss_xentropy

        
        
        # Compute the total loss for the batch and add it to train_loss
        train_loss += loss.item() * inputs.size(0)
        
        train_loss_contrastive += loss_contrastive.item() * inputs.size(0)
        train_loss_xentropy += loss_xentropy.item() * inputs.size(0)

        # Compute the accuracy
        ret, predictions = torch.max(outputs.data, 1)
        correct_counts = predictions.eq(labels.data.view_as(predictions))

        # Convert correct_counts to float and then compute the mean
        acc = torch.mean(correct_counts.type(torch.FloatTensor))

        # Compute total accuracy in the whole batch and add to train_acc
#         print(loss.item(), inputs.size(0))
        train_acc += acc.item() * inputs.size(0)
    
        # Backpropagate the gradients
        loss.backward()

        # Update the parameters
        optimizer.step()
        if scheduler:
            scheduler.step()
        
#         print(loss_contrastive.item())
#         print(loss_xentropy.item())
        if tracking:  
            wandb.log({"batch_train_loss_contrastive": loss_contrastive.item()})
            wandb.log({"batch_train_loss_xentropy": loss_xentropy.item()})
            
            wandb.log({"batch_train_loss": loss.item()})
            wandb.log({"batch_train_acc": acc.item()}) 
    
#         print(loss_contrastive.item(), loss_xentropy.item(), loss.item(), acc.item())
        
    # Find average training loss and training accuracy
    avg_train_loss = train_loss/train_data_size 
    avg_train_loss_contrastive = train_loss_contrastive/train_data_size 
    avg_train_loss_xentropy = train_loss_xentropy/train_data_size 

    avg_train_acc = train_acc/float(train_data_size)
    print("Training: Loss: {:.4f},  contrastive: {:.4f}, crossentropy: {:.4f}, Accuracy: {:.4f}%".format(
            avg_train_loss, avg_train_loss_contrastive, avg_train_loss_xentropy, avg_train_acc*100))

    return avg_train_loss, avg_train_loss_contrastive, avg_train_loss_xentropy, avg_train_acc

        
    

def valid(model, loss_func_list, device, valid_dl, process_func, test=False,
          tracking=False):
    
    valid_data_size = len(valid_dl.dataset)
    model = model.to(device)

    # Loss and Accuracy within the epoch
    valid_loss = 0.0
    valid_loss_contrastive = 0.0
    valid_loss_xentropy = 0.0    
    valid_acc = 0.0
     
    # Validation - No gradient tracking needed
    with torch.no_grad():

        # Set to evaluation mode
        model.eval()

        # Validation loop
        for j, batch in enumerate(tqdm(valid_dl)):

            if test:
                
                inputs, labels = process_func.process(batch)
                outputs_fpn, outputs = model(inputs)
                loss = loss_func_list[1](outputs, labels)
                
            else:
                [inputs, inputs_p], labels = process_func.process(batch)

                # loss - contrasitve
                outputs_fpn, outputs = model(inputs)
                outputs_p_fpn, outputs_ = model(inputs_p)
                loss_contrastive = loss_func_list[0](outputs_fpn, outputs_p_fpn) 

                # loss 2
                loss_xentropy = loss_func_list[1](outputs, labels)

                loss = loss_contrastive + loss_xentropy
            
            # Compute the total loss for the batch and add it to valid_loss
            valid_loss += loss.item() * len(inputs)
            
            if not test:
                valid_loss_contrastive += loss_contrastive.item() * inputs.size(0)
                valid_loss_xentropy += loss_xentropy.item() * inputs.size(0)

            # Calculate validation accuracy
            ret, predictions = torch.max(outputs.data, 1)
            correct_counts = predictions.eq(labels.data.view_as(predictions))

            # Convert correct_counts to float and then compute the mean
            acc = torch.mean(correct_counts.type(torch.FloatTensor))

            # Compute total accuracy in the whole batch and add to valid_acc
            valid_acc += acc.item() * inputs.size(0) 
            
            if tracking:  

                if test:
                    wandb.log({"batch_test_loss": loss.item()})
                    wandb.log({"batch_test_acc": acc.item()})  
                
                else:
                    wandb.log({"batch_valid_loss_contrastive": loss_contrastive.item()})
                    wandb.log({"batch_valid_loss_xentropy": loss_xentropy.item()})
                    wandb.log({"batch_valid_loss": loss.item()})
                    wandb.log({"batch_valid_acc": acc.item()})            
              
    if test:
        # Find average training loss and training accuracy
        avg_valid_loss = valid_loss/valid_data_size 
        avg_valid_acc = valid_acc/float(valid_data_size)

        print("Cross validation: Loss: {:.4f}, Accuracy: {:.4f}%".format(
            avg_valid_loss, avg_valid_acc*100))
        
        return avg_valid_loss, avg_valid_acc
            
            
    else:
        # Find average training loss and training accuracy
        avg_valid_loss = valid_loss/valid_data_size 
        avg_valid_loss_contrastive = valid_loss_contrastive/valid_data_size 
        avg_valid_loss_xentropy = valid_loss_xentropy/valid_data_size 
        avg_valid_acc = valid_acc/float(valid_data_size)

        print("Validation: Loss: {:.4f},  contrastive: {:.4f}, crossentropy: {:.4f}, Accuracy: {:.4f}%".format(
                avg_valid_loss, avg_valid_loss_contrastive, avg_valid_loss_xentropy, avg_valid_acc*100))
        
        return avg_valid_loss, avg_valid_loss_contrastive, avg_valid_loss_xentropy, avg_valid_acc

In [6]:
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

class fpn_resnet50(nn.Module):
    def __init__(self, backbone):
        super(fpn_resnet50, self).__init__() # Initialize self._modules as OrderedDict

        self.backbone = backbone
        
        self.classifier = nn.Sequential(  
                        nn.BatchNorm1d(256),
                        nn.Dropout(p=0.5, inplace=False),
                        nn.Linear(in_features = 256, out_features=64, bias=False),
                        nn.ReLU(inplace=True), 

                        nn.BatchNorm1d(64),
                        nn.Dropout(p=0.5, inplace=False),
                        nn.Linear(in_features=64, out_features=5, bias=True)
                        )
            
    def merge_orderdict(self, x):
        return torch.cat(( nn.AdaptiveAvgPool2d(output_size=(1))(x['3']), 
                           nn.AdaptiveAvgPool2d(output_size=(1))(x['pool'])), axis=1).squeeze()
    
    def forward(self, x):
        x = self.backbone(x)
        x_acc = x['pool']
#         x_fpn = x
        x_fpn = self.merge_orderdict(x)
#         x_fpn = nn.AdaptiveAvgPool2d(output_size=(1))(x_fpn).squeeze()

        x_acc = nn.AdaptiveAvgPool2d(output_size=(1))(x_acc).squeeze()
        x_acc = self.classifier(x_acc)        
        return x_fpn, x_acc

backbone = resnet_fpn_backbone('resnet50', pretrained=True, trainable_layers=3)
model = fpn_resnet50(backbone)

In [ ]:
resnet_fpn_backbone('resnet50', pretrained=False, trainable_layers=3)

In [15]:
model

fpn_resnet50(
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
          (relu): ReLU(inplace=True)
          (downsample): Sequential(
            (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): FrozenBatchNorm2d(256

In [ ]:
out = backbone(torch.randn(2,3,256,256))

In [13]:
len(out), out.keys()

(5, odict_keys(['0', '1', '2', '3', 'pool']))

In [14]:
for k in out.keys():
    print(out[k].shape)

torch.Size([2, 256, 64, 64])
torch.Size([2, 256, 32, 32])
torch.Size([2, 256, 16, 16])
torch.Size([2, 256, 8, 8])
torch.Size([2, 256, 4, 4])


In [6]:
class ContrastiveLoss(torch.nn.Module):
    """
    Contrastive loss function.
    Based on: http://yann.lecun.com/exdb/publis/pdf/hadsell-chopra-lecun-06.pdf;
              https://towardsdatascience.com/how-to-choose-your-loss-when-designing-a-siamese-neural-net-contrastive-triplet-or-quadruplet-ecba11944ec
              https://innovationincubator.com/siamese-neural-network-with-pytorch-code-example/
    """    
    def __init__(self, n):
        super(ContrastiveLoss, self).__init__()
        self.n = n
        
    def forward(self, output1, output2):
        # Find the pairwise distance or eucledian distance of two output feature vectors
        dist = F.pairwise_distance(output1, output2)
        # perform contrastive loss calculation with the distance
        loss = torch.mean(self.n * torch.pow(dist, 2))
        return loss



class IncreasingBalanceRatio():
    '''
    coef is the ratio of ContrastiveLoss/CrossEntropyLoss,
    start_coef: coef at the start,
    end_coef: coef at the end,
    epochs: total epochs,
    order: how coef increase with epochs,
    '''

    def __init__(self, start_coef, end_coef, start_epoch, epochs, order):
        self.start_coef = start_coef
        self.end_coef = end_coef
        self.start_epoch = start_epoch
        self.epochs = epochs
        self.order = order

    def IncreasingRatio(self):
        coef_list = self.start_coef+np.linspace(0, 1, self.epochs)**self.order*(self.end_coef-self.start_coef)
        return coef_list

    def LossFunction(self, epoch_index, prev_coef_list, prev_contrastive, prev_xentropy):
#         print(prev_coef_list)
        n = np.min([len(prev_coef_list), 5])
        prev_coef = np.mean(prev_coef_list[-n:])
        coef_list = self.IncreasingRatio()
        coef = coef_list[epoch_index-self.start_epoch]
#         print(coef_list, len(coef_list))
        balance_factor = (prev_contrastive * coef) / (prev_xentropy * prev_coef)
        return ContrastiveLoss(balance_factor), balance_factor, coef

In [7]:
# start = 0
# epochs = 100

# start_coef, end_coef, order = 0.01, 1, 1
# loss_func_list = [IncreasingBalanceRatio(start_coef, end_coef, start, epochs, order), 
#                   nn.CrossEntropyLoss()]

# epoch_index = 99
# prev_coef = [1]
# prev_contrastive = 1e-3
# prev_xentropy = 1e-3

# loss_func_list[0].LossFunction(epoch_index, prev_coef, prev_contrastive, prev_xentropy)

### start_coef, end_coef, order = 1e-3, 5e-3, 1

In [7]:
config = {
    'batch_size': bs,
    'optimizer': 'Adam',
    'scheduler': 'OneCycleLR',
    'dropout': 0.5
}

NAME = 'start_coef=1e-3, end_coef=5e-3, order=1'

wandb.init(project='Symmetry-contrastive_loss', entity='yig319', 
           name=NAME, config=config, reinit=True, id='16uzk0oz')
config = wandb.config
print(wandb.run.id)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: yig319 (use `wandb login --relogin` to force relogin)
wandb: wandb version 0.12.16 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade


16uzk0oz


In [9]:
start = 0
epochs = 20
lr = 1e-3
device = torch.device('cuda:1')

start_coef, end_coef, order = 1e-3, 5e-3, 1
loss_func_list = [IncreasingBalanceRatio(start_coef, end_coef, start, epochs, order), 
                  nn.CrossEntropyLoss()]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, 
                            steps_per_epoch=len(train_dl))
proc = process_function(0,1, device=device)    
tracking = True

history = train_epochs(model, loss_func_list, optimizer, device, train_dl, valid_dl, test_dl,
                       start=start, epochs=epochs, process_func=proc, scheduler=scheduler, 
                       model_name=NAME, model_dir='/scratch/yichen/models/', tracking=tracking)

Epoch: 1/20


100%|██████████| 81/81 [14:34<00:00, 10.79s/it] 


Training: Loss: 170.7914,  contrastive: 169.0354, crossentropy: 1.7559, Accuracy: 24.2040%


100%|██████████| 17/17 [01:20<00:00,  4.72s/it]


Validation: Loss: 3.6322,  contrastive: 2.1005, crossentropy: 1.5316, Accuracy: 32.6000%


...

100%|██████████| 17/17 [01:19<00:00,  4.70s/it]


Validation: Loss: 0.3904,  contrastive: 0.0000, crossentropy: 0.3904, Accuracy: 87.9000%


100%|██████████| 17/17 [00:40<00:00,  2.39s/it]


Cross validation: Loss: 1.9237, Accuracy: 27.7200%


wandb: Network error resolved after 0:00:59.320896, resuming normal operation.


In [10]:
start = 20
epochs = 80
lr = 1e-3
device = torch.device('cuda:1')

start_coef, end_coef, order = 5e-3, 1e-1, 1
loss_func_list = [IncreasingBalanceRatio(start_coef, end_coef, start, epochs, order), 
                  nn.CrossEntropyLoss()]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, 
                            steps_per_epoch=len(train_dl))
proc = process_function(0,1, device=device)    
tracking = True

history = train_epochs(model, loss_func_list, optimizer, device, train_dl, valid_dl, test_dl,
                       start=start, epochs=epochs, process_func=proc, scheduler=scheduler, 
                       model_name=NAME, model_dir='/scratch/yichen/models/', tracking=tracking)

Epoch: 21/100


100%|██████████| 81/81 [14:37<00:00, 10.83s/it] 


Training: Loss: 1.9123,  contrastive: 0.9273, crossentropy: 0.9850, Accuracy: 61.3440%


100%|██████████| 17/17 [01:20<00:00,  4.76s/it]


Validation: Loss: 5.6954,  contrastive: 0.1413, crossentropy: 5.5542, Accuracy: 24.4200%


...

Validation: Loss: 0.0666,  contrastive: 0.0002, crossentropy: 0.0664, Accuracy: 97.7400%


100%|██████████| 17/17 [00:40<00:00,  2.40s/it]


Cross validation: Loss: 2.1107, Accuracy: 42.1800%
Epoch: 57/100


 35%|███▍      | 28/81 [05:44<10:52, 12.31s/it] 


KeyboardInterrupt: 

In [9]:
NAME

'start_coef=1e-3, end_coef=5e-3, order=1'

In [10]:
model = torch.load('/scratch/yichen/models/start_coef=1e-3, end_coef=5e-3, order=1-epoch-50.pt')

In [11]:
start = 50
epochs = 50
lr = 1e-3
device = torch.device('cuda:1')

start_coef, end_coef, order = 5e-2, 1, 1
loss_func_list = [IncreasingBalanceRatio(start_coef, end_coef, start, epochs, order), 
                  nn.CrossEntropyLoss()]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, 
                            steps_per_epoch=len(train_dl))
proc = process_function(0,1, device=device)    
tracking = True

history = train_epochs(model, loss_func_list, optimizer, device, train_dl, valid_dl, test_dl,
                       start=start, epochs=epochs, process_func=proc, scheduler=scheduler, 
                       model_name=NAME, model_dir='/scratch/yichen/models/', tracking=tracking)

Epoch: 51/100


100%|██████████| 81/81 [15:04<00:00, 11.16s/it] 


Training: Loss: 2.7790,  contrastive: 2.1761, crossentropy: 0.6030, Accuracy: 78.3280%


100%|██████████| 17/17 [01:22<00:00,  4.88s/it]


Validation: Loss: 19.9083,  contrastive: 0.2774, crossentropy: 19.6309, Accuracy: 20.0000%


...

Cross validation: Loss: 5.1945, Accuracy: 27.4800%
Epoch: 85/100


100%|██████████| 81/81 [15:56<00:00, 11.81s/it] 


Training: Loss: 0.0240,  contrastive: 0.0000, crossentropy: 0.0239, Accuracy: 99.3840%


 47%|████▋     | 8/17 [00:55<01:01,  6.89s/it]


KeyboardInterrupt: 

### start_coef, end_coef, order = 1e-5, 1e-2, 3

In [8]:
config = {
    'batch_size': bs,
    'optimizer': 'Adam',
    'scheduler': 'OneCycleLR',
    'dropout': 0.5
}

NAME = 'start_coef=1e-5, end_coef=1e-2, order=3'

wandb.init(project='Symmetry-contrastive_loss', entity='yig319', 
           name=NAME, config=config, reinit=True)
config = wandb.config
print(wandb.run.id)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: yig319 (use `wandb login --relogin` to force relogin)
wandb: wandb version 0.12.16 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade


2gglsnz4


In [9]:
start = 0
epochs = 100
lr = 1e-3
device = torch.device('cuda:1')

start_coef, end_coef, order = 1e-5, 1e-2, 3
loss_func_list = [IncreasingBalanceRatio(start_coef, end_coef, start, epochs, order), 
                  nn.CrossEntropyLoss()]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, 
                            steps_per_epoch=len(train_dl))
proc = process_function(0,1, device=device)    
tracking = True

history = train_epochs(model, loss_func_list, optimizer, device, train_dl, valid_dl, test_dl,
                       start=start, epochs=epochs, process_func=proc, scheduler=scheduler, 
                       model_name=NAME, model_dir='/scratch/yichen/models/', tracking=tracking)

Epoch: 1/100


100%|██████████| 81/81 [15:41<00:00, 11.62s/it] 


Training: Loss: 3.0000,  contrastive: 1.6942, crossentropy: 1.3058, Accuracy: 46.6560%


100%|██████████| 17/17 [01:24<00:00,  4.98s/it]


Validation: Loss: 1.1466,  contrastive: 0.0788, crossentropy: 1.0678, Accuracy: 54.9400%


...

Validation: Loss: 48007.1538,  contrastive: 47968.4269, crossentropy: 38.7268, Accuracy: 20.0000%


100%|██████████| 17/17 [00:43<00:00,  2.53s/it]


Cross validation: Loss: 38.6619, Accuracy: 20.0000%
Epoch: 7/100


  1%|          | 1/81 [00:47<1:02:49, 47.12s/it]


KeyboardInterrupt: 

### start_coef, end_coef, order = 0.01, 0.1, 2

In [8]:
config = {
    'batch_size': bs,
    'optimizer': 'Adam',
    'scheduler': 'OneCycleLR',
    'dropout': 0.5
}

NAME = 'start_coef=0.01, end_coef=0.1, order=2'

wandb.init(project='Symmetry-contrastive_loss', entity='yig319', 
           name=NAME, config=config, reinit=True)
config = wandb.config
print(wandb.run.id)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: yig319 (use `wandb login --relogin` to force relogin)
wandb: wandb version 0.12.16 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade


2sb71d7c


In [10]:
start = 0
epochs = 100
lr = 1e-3
device = torch.device('cuda:1')

start_coef, end_coef, order = 0.01, 0.1, 2
loss_func_list = [IncreasingBalanceRatio(start_coef, end_coef, start, epochs, order), 
                  nn.CrossEntropyLoss()]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, 
                            steps_per_epoch=len(train_dl))
proc = process_function(0,1, device=device)    
tracking = True

history = train_epochs(model, loss_func_list, optimizer, device, train_dl, valid_dl, test_dl,
                       start=start, epochs=epochs, process_func=proc, scheduler=scheduler, 
                       model_name=NAME, model_dir='/scratch/yichen/models/', tracking=tracking)

Epoch: 1/100


100%|██████████| 81/81 [14:22<00:00, 10.65s/it] 


Training: Loss: 0.6439,  contrastive: 0.0269, crossentropy: 0.6170, Accuracy: 80.9680%


100%|██████████| 17/17 [01:21<00:00,  4.81s/it]


Validation: Loss: 0.2970,  contrastive: 0.0157, crossentropy: 0.2812, Accuracy: 93.3000%


...

Training: Loss: 77.4405,  contrastive: 75.8303, crossentropy: 1.6103, Accuracy: 19.7360%


100%|██████████| 17/17 [01:21<00:00,  4.79s/it]


Validation: Loss: 162.1109,  contrastive: 64.5212, crossentropy: 97.5897, Accuracy: 20.0000%


  0%|          | 0/17 [00:00<?, ?it/s]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

 71%|███████   | 12/17 [01:08<00:28,  5.74s/it]


KeyboardInterrupt: 

### start_coef, end_coef, order = 0.01, 1, 2

In [9]:
config = {
    'batch_size': bs,
    'optimizer': 'Adam',
    'scheduler': 'OneCycleLR',
    'dropout': 0.5
}

NAME = 'resnet34-pair_ds-pyramid_contrastive_loss-IncreasingBalanceRatio'

wandb.init(project='Symmetry-contrastive_loss', entity='yig319', 
           name=NAME, config=config, reinit=True)
config = wandb.config
print(wandb.run.id)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: yig319 (use `wandb login --relogin` to force relogin)
wandb: wandb version 0.12.16 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade


3kqblg9k


In [10]:
start = 0
epochs = 100
lr = 1e-3
device = torch.device('cuda:1')

start_coef, end_coef, order = 0.01, 1, 2
loss_func_list = [IncreasingBalanceRatio(start_coef, end_coef, start, epochs, order), 
                  nn.CrossEntropyLoss()]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, 
                            steps_per_epoch=len(train_dl))
proc = process_function(0,1, device=device)    
tracking = True

history = train_epochs(model, loss_func_list, optimizer, device, train_dl, valid_dl, test_dl,
                       start=start, epochs=epochs, process_func=proc, scheduler=scheduler, 
                       model_name=NAME, model_dir='/scratch/yichen/models/', tracking=tracking)

Epoch: 1/100


100%|██████████| 81/81 [15:10<00:00, 11.25s/it] 


Training: Loss: 0.6312,  contrastive: 0.0260, crossentropy: 0.6052, Accuracy: 80.3640%


100%|██████████| 17/17 [01:22<00:00,  4.86s/it]


Validation: Loss: 0.2450,  contrastive: 0.0168, crossentropy: 0.2283, Accuracy: 93.6400%


...

Training: Loss: 21.2751,  contrastive: 19.6652, crossentropy: 1.6100, Accuracy: 20.0160%


100%|██████████| 17/17 [01:16<00:00,  4.49s/it]


Validation: Loss: 71.8184,  contrastive: 19.7325, crossentropy: 52.0859, Accuracy: 20.0000%


 94%|█████████▍| 16/17 [00:39<00:01,  1.41s/it]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

 96%|█████████▋| 78/81 [14:27<00:33, 11.13s/it]


KeyboardInterrupt: 